In [1]:
from sedona.spark import SedonaContext
from sedona.maps.SedonaKepler import SedonaKepler
import os
from shapely.wkt import dumps
from shapely.geometry import Polygon

/tmp/ipykernel_318/3243136233.py:2: DeprecationWarning: The 'sedona.maps' module is deprecated and will be removed in future versions. Please use 'sedona.spark.maps' instead.
  from sedona.maps.SedonaKepler import SedonaKepler


In [2]:
%%capture
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder()

sedona = SedonaContext.create(config.getOrCreate())
sedona.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/06 23:37:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/09/06 23:37:27 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/09/06 23:37:27 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/09/06 23:37:27 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/09/06 23:37:27 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
25/09/06 23:37:27 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/09/06 23:37:27 WARN SimpleFunctionRegistry: The function st_envelop

# Read data from bucket

In [3]:
nyx_taxi = sedona\
    .read\
    .format("parquet")\
    .load(f"s3a://{bucket_name}/source_data/nyc_yellow")

# Explore the data

In [4]:
explore = nyx_taxi\
    .where("pickup_longitude IS NOT NULL AND pickup_latitude IS NOT NULL")\
    .where("dropoff_longitude IS NOT NULL AND dropoff_latitude IS NOT NULL")\
    .selectExpr(
        "ST_POINT(CAST(pickup_longitude AS DOUBLE), CAST(pickup_latitude AS DOUBLE)) AS geom"
    ).sample(0.00001)

In [5]:
explore.cache().count()

2187

In [6]:
SedonaKepler.create_map(explore, "exploration")

/usr/local/lib/python3.10/dist-packages/keplergl/keplergl.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_string


KeplerGl(data={'exploration': {'index': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,…

# transform the data to sedona spatial dataframe

In [7]:
nyx_taxi_with_coordinates = nyx_taxi\
    .where("pickup_longitude IS NOT NULL AND pickup_latitude IS NOT NULL")\
    .where("dropoff_longitude IS NOT NULL AND dropoff_latitude IS NOT NULL")\
    .selectExpr(
        "CAST(total_amount AS DECIMAL(24, 10)) AS total_amount",
        "ST_POINT(CAST(pickup_longitude AS DOUBLE), CAST(pickup_latitude AS DOUBLE)) AS pickup_geom",
        "ST_POINT(CAST(dropoff_longitude AS DOUBLE), CAST(dropoff_latitude AS DOUBLE)) AS dropoff_geom"
    ).createOrReplaceTempView("taxi")

In [8]:
polygon = {"type":"Polygon","coordinates":[[[-74.11902177180524,40.83533792558916],[-74.22186537559533,40.649655372448066],[-73.75749417169013,40.60368516322427],[-73.73730411934669,40.69323888136599],[-73.8988245380961,40.90133870775822],[-74.12557435672555,40.914249930541786],[-74.11780895197751,40.83556976599659],[-74.11902177180524,40.83533792558916]]]}

In [9]:
polygon_wkt = dumps(Polygon(polygon["coordinates"][0]))

In [10]:
# filter to a given polygon

In [11]:
sedona.sql(
f"""
    SELECT * 
    FROM taxi
    WHERE ST_Within(dropoff_geom, ST_GeomFromText('{polygon_wkt}')) 
    AND ST_Within(pickup_geom, ST_GeomFromText('{polygon_wkt}'))
    AND total_amount > 0
"""
).createOrReplaceTempView("taxi_cleaned")

# Find most popular Pickup points

In [12]:
most_popular_pickup = sedona.sql(
    """
    WITH h3_index AS (
        SELECT 
            ST_H3CellIDs(pickup_geom, 8, true)[0] AS h3_id 
        FROM taxi_cleaned
    )
    SELECT 
        h3_id,
        count(h3_id) AS cnt,
        ST_H3ToGeom(array(h3_id))[0] AS geom
    FROM h3_index
    GROUP BY h3_id
    ORDER BY count(h3_id) DESC
    """
)

In [13]:
most_popular_pickup.cache()

DataFrame[h3_id: bigint, cnt: bigint, geom: udt]

In [14]:
most_popular_pickup.count()

1215

In [15]:
SedonaKepler.create_map(most_popular_pickup, "most-popular-pickup")

KeplerGl(data={'most-popular-pickup': {'index': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17,…

# Find most popular routes

In [16]:
most_popular_routes = sedona.sql("""
    WITH h3_indexes AS (
        SELECT 
            ST_H3CellIDs(pickup_geom, 8, true)[0] AS h3_pickup_id,
            ST_H3CellIDs(dropoff_geom, 8, true)[0] AS h3_dropoff_id
        FROM taxi_cleaned
    ),
    indexed AS (
        SELECT 
            CASE 
                WHEN h3_pickup_id > h3_dropoff_id 
                THEN CONCAT(h3_dropoff_id, ' ', h3_pickup_id) 
                ELSE CONCAT(h3_pickup_id, ' ', h3_dropoff_id) 
            END AS id,
            h3_pickup_id,
            h3_dropoff_id
        FROM h3_indexes
    ),
    groupped AS (
        SELECT 
            id,
            count(id) AS cnt,
            first(h3_pickup_id) AS h3_pickup_id,
            first(h3_dropoff_id) AS h3_dropoff_id
        FROM indexed
        GROUP BY id
    )
    SELECT 
        h3_pickup_id,
        h3_dropoff_id,
        ST_MakeLine(
            ST_Centroid(ST_H3ToGeom(array(h3_pickup_id))[0]),
            ST_Centroid(ST_H3ToGeom(array(h3_dropoff_id))[0])
        ) AS geom,
        cnt
    FROM groupped
    ORDER BY cnt DESC
    LIMIT 20

""")

In [30]:
most_popular_routes.cache()

DataFrame[h3_p_id: bigint, h3_d_id: bigint, geom: udt, cnt: bigint]

In [31]:
most_popular_routes.count()

20

In [17]:
SedonaKepler.create_map(most_popular_routes, "Most popular routes")

KeplerGl(data={'Most popular routes': {'index': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17,…